### 1. Import Necessary libraries

In [2]:
import pandas as pd
import keras
from sklearn.preprocessing import StandardScaler

## 2. Import Data

In [3]:
diabetes_data = pd.read_csv('diabetes.csv')
diabetes_data

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


## 3. Data Preparation

In [11]:
X = diabetes_data.drop('Outcome',axis = 1)
y = diabetes_data[['Outcome']]

In [12]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state=12,stratify=y)

In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)  
X_test = scaler.transform(X_test)   

### HYPERPARAMETER TUNING

#### List the Hyperparamaters we used so far....
1. Learning Rate
2. Epochs
3. Batches
4. Activation Functions
5. Optimizers
6. Loss Functions
7. Kernel Initializers

### ========================================================

In [5]:
!pip install scikeras


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Hyperparameter all at once
#### This process is computationally expensive.

In [34]:
import numpy as np
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

from scikeras.wrappers import KerasClassifier 
##I am building a neural network model using Keras Functional API, and I am using GridSearchCV with KerasClassifier to find the best parameters for the model
from sklearn.model_selection import GridSearchCV, KFold

In [47]:
def create_model(learning_rate=0.001,dropout_rate=0.0,activation_function='relu', init='glorot_uniform',neuron1=8, neuron2=4):

    model = Sequential()
    model.add(Dense(neuron1,input_shape=(8,),kernel_initializer=init,activation=activation_function))
    model.add(Dropout(dropout_rate))
    model.add(Dense(neuron2,kernel_initializer=init,activation=activation_function))    
    model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation='sigmoid'))
    
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(loss='binary_crossentropy',optimizer=optimizer,metrics=['accuracy'])
    return model

In [48]:
model = KerasClassifier(model=create_model,verbose=1)

In [49]:
param_grid = {'model__learning_rate': [0.001, 0.01], 
               #if its model_lr = Just variable naming, but here it is model__learning_rate -> Navigate inside object
              'model__dropout_rate': [0.0, 0.2],
              'model__activation_function': ['relu', 'tanh'],
              'model__init': ['glorot_uniform', 'he_uniform'],
              'model__neuron1': [8, 16],
              'model__neuron2': [4, 8],
              'batch_size': [10, 20],
              'epochs': [10, 50]
             }

In [51]:
kfold = KFold(n_splits=3, shuffle=True, random_state=42)
grid = GridSearchCV(estimator=model,param_grid=param_grid,cv=kfold,verbose=10,n_jobs=-1)   # parallel processing 🔥

In [43]:
grid_result = grid.fit(X_train, y_train)

Fitting 3 folds for each of 256 candidates, totalling 768 fits


In [44]:
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']

for mean, std, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, std, param))

Best: 0.785039 using {'batch_size': 20, 'epochs': 10, 'model__activation_function': 'relu', 'model__dropout_rate': 0.0, 'model__init': 'glorot_uniform', 'model__learning_rate': 0.01, 'model__neuron1': 8, 'model__neuron2': 8}
0.718309 (0.031681) with: {'batch_size': 10, 'epochs': 10, 'model__activation_function': 'relu', 'model__dropout_rate': 0.0, 'model__init': 'glorot_uniform', 'model__learning_rate': 0.001, 'model__neuron1': 8, 'model__neuron2': 4}
0.739471 (0.027507) with: {'batch_size': 10, 'epochs': 10, 'model__activation_function': 'relu', 'model__dropout_rate': 0.0, 'model__init': 'glorot_uniform', 'model__learning_rate': 0.001, 'model__neuron1': 8, 'model__neuron2': 8}
0.742643 (0.014511) with: {'batch_size': 10, 'epochs': 10, 'model__activation_function': 'relu', 'model__dropout_rate': 0.0, 'model__init': 'glorot_uniform', 'model__learning_rate': 0.001, 'model__neuron1': 16, 'model__neuron2': 4}
0.729547 (0.041430) with: {'batch_size': 10, 'epochs': 10, 'model__activation_fun

In [45]:
results_df = pd.DataFrame(grid_result.cv_results_)
results_df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_batch_size,param_epochs,param_model__activation_function,param_model__dropout_rate,param_model__init,param_model__learning_rate,param_model__neuron1,param_model__neuron2,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score
0,5.045100,0.195312,0.493346,0.029431,10,10,relu,0.0,glorot_uniform,0.001,8,4,"{'batch_size': 10, 'epochs': 10, 'model__activ...",0.712195,0.682927,0.759804,0.718309,0.031681,228
1,5.440703,0.433157,0.464653,0.023351,10,10,relu,0.0,glorot_uniform,0.001,8,8,"{'batch_size': 10, 'epochs': 10, 'model__activ...",0.736585,0.707317,0.774510,0.739471,0.027507,185
2,4.871217,0.245826,0.490717,0.033397,10,10,relu,0.0,glorot_uniform,0.001,16,4,"{'batch_size': 10, 'epochs': 10, 'model__activ...",0.741463,0.760976,0.725490,0.742643,0.014511,181
3,5.543427,0.647505,0.474578,0.018647,10,10,relu,0.0,glorot_uniform,0.001,16,8,"{'batch_size': 10, 'epochs': 10, 'model__activ...",0.751220,0.765854,0.671569,0.729547,0.041430,208
4,5.541381,0.638789,0.447228,0.018382,10,10,relu,0.0,glorot_uniform,0.010,8,4,"{'batch_size': 10, 'epochs': 10, 'model__activ...",0.741463,0.741463,0.759804,0.747577,0.008646,160
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251,19.563561,0.354163,0.258666,0.008562,20,50,tanh,0.2,he_uniform,0.001,16,8,"{'batch_size': 20, 'epochs': 50, 'model__activ...",0.756098,0.731707,0.764706,0.750837,0.013976,144
252,18.166126,0.287159,0.515541,0.246011,20,50,tanh,0.2,he_uniform,0.010,8,4,"{'batch_size': 20, 'epochs': 50, 'model__activ...",0.731707,0.736585,0.769608,0.745967,0.016835,164
253,17.083828,0.414435,0.243767,0.005796,20,50,tanh,0.2,he_uniform,0.010,8,8,"{'batch_size': 20, 'epochs': 50, 'model__activ...",0.780488,0.741463,0.759804,0.760585,0.015941,81
254,16.634895,0.406208,0.148231,0.005946,20,50,tanh,0.2,he_uniform,0.010,16,4,"{'batch_size': 20, 'epochs': 50, 'model__activ...",0.770732,0.731707,0.774510,0.758983,0.019348,90


In [46]:
results_df.to_csv("results.csv")

## THE END!!